In [46]:
# Install dependencies
!pip install langchain langgraph pytesseract pdf2image PyPDF2 pillow transformers torch torchvision sentencepiece
!pip install opencv-python-headless



In [47]:
#imports
import os, datetime, sqlite3, pytesseract
from pdf2image import convert_from_path
from PyPDF2 import PdfReader
from PIL import Image
from transformers import pipeline
from tabulate import tabulate
import numpy as np
import cv2



# --- Initialize Models ---
text_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
image_classifier = pipeline("image-classification", model="google/vit-base-patch16-224")



Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [48]:
class DBTools:
    def __init__(self, db_path="cease_desist.db"):
        self.db_path = db_path

    def init_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""CREATE TABLE IF NOT EXISTS documents (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date_received TEXT,
            document_name TEXT,
            details TEXT,
            document_type TEXT,
            human_answer TEXT,
            related_doc_id INTEGER,
            conversation_context TEXT)""")
        conn.commit()
        conn.close()

    def store(self, doc_name, details, doc_type, human_answer=None,
              related_doc_id=None, conversation_context=None):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO documents (date_received, document_name, details, document_type, human_answer, related_doc_id, conversation_context) VALUES (?, ?, ?, ?, ?, ?, ?)",
            (str(datetime.date.today()), doc_name, details[:200], doc_type, human_answer, related_doc_id, conversation_context)
        )
        conn.commit()
        conn.close()

    def recall(self, doc_name):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM documents WHERE document_name=?", (doc_name,))
        row = cursor.fetchone()
        conn.close()
        return row

    def query_all(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM documents")
        rows = cursor.fetchall()
        headers = [desc[0] for desc in cursor.description]
        conn.close()
        return headers, rows


In [ ]:
'''def load_document(file_path, lang="eng"):
    text, images = "", []
    # Try extracting embedded text
    try:
        reader = PdfReader(file_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
    except:
        pass

    # Convert pages to images with higher DPI
    try:
        pages = convert_from_path(file_path, dpi=300)
        images.extend(pages)
        for img in pages:
            # Preprocess for better OCR
            gray = img.convert("L")  # grayscale
            # Optional: thresholding for noisy scans
            # gray = gray.point(lambda x: 0 if x < 128 else 255, '1')
            ocr_text = pytesseract.image_to_string(gray, lang=lang)
            if ocr_text.strip():
                text += "\n" + ocr_text
    except Exception as e:
        print("OCR error:", e)

    return text, images
'''

In [49]:
def preprocess_for_ocr(pil_img):
    # Convert PIL image to OpenCV format
    img = np.array(pil_img)
    if len(img.shape) == 3:  # color image
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Apply thresholding (binarization)
    _, img = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY)
    # Optional: deskew or denoise if needed
    return img


In [50]:
def load_document(file_path, lang="eng"):
    text, images = "", []
    # Try extracting embedded text
    try:
        reader = PdfReader(file_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
    except:
        pass

    # Convert pages to images with higher DPI
    try:
        pages = convert_from_path(file_path, dpi=300)
        images.extend(pages)
        for img in pages:
            proc_img = preprocess_for_ocr(img)
            ocr_text = pytesseract.image_to_string(proc_img, lang=lang)
            if ocr_text.strip():
                text += "\n" + ocr_text
    except Exception as e:
        print("OCR error:", e)

    return text, images


In [51]:
# --- Step 2: Confidence-based Label Assignment ---
def assign_label(scores, threshold=0.75, margin=0.10):
    top_label = max(scores, key=scores.get)
    top_score = scores[top_label]

    if top_score >= threshold:
        return top_label
    else:
        sorted_scores = sorted(scores.values(), reverse=True)
        if sorted_scores[0] - sorted_scores[1] < margin:
            return "Uncertain"
        return top_label


In [52]:
# --- Classification Agent ---
def classify_multimodal(text, images, return_confidence=False):
    labels = ["Cease", "Uncertain", "Irrelevant"]
    scores = {}

    # Text classification
    if text.strip():
        text_result = text_classifier(text, candidate_labels=labels)
        for lbl, sc in zip(text_result["labels"], text_result["scores"]):
            scores[lbl] = max(scores.get(lbl, 0), sc)

    # Image classification (OCR + fallback)
    if images:
        ocr_text = "".join([pytesseract.image_to_string(img) for img in images])
        if ocr_text.strip():
            img_text_result = text_classifier(ocr_text, candidate_labels=labels)
            for lbl, sc in zip(img_text_result["labels"], img_text_result["scores"]):
                scores[lbl] = max(scores.get(lbl, 0), sc)
        else:
            img_result = image_classifier(images[0])
            scores["Irrelevant"] = max(scores.get("Irrelevant", 0), img_result[0]["score"])

    # If no scores at all
    if not scores:
        classification = "Uncertain"
    else:
        classification = assign_label(scores)

    # Return both label and scores if requested
    if return_confidence:
        return classification, scores
    return classification


In [ ]:
# --- Step 4: Database Agent ---
'''def init_db():
    conn = sqlite3.connect("cease_desist.db")
    cursor = conn.cursor()
    cursor.execute("""CREATE TABLE IF NOT EXISTS cease_requests (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_received TEXT,
        document_name TEXT,
        details TEXT,
        document_type TEXT,
        human_answer TEXT)""")
    conn.commit()
    conn.close()'''



In [ ]:
'''def store_in_db(doc_name, details, doc_type, human_answer=None):
    conn = sqlite3.connect("cease_desist.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO cease_requests (date_received, document_name, details, document_type, human_answer) VALUES (?, ?, ?, ?, ?)",
                   (str(datetime.date.today()), doc_name, details[:200], doc_type, human_answer))
    conn.commit()
    conn.close()'''


In [53]:
# --- Step 5: Archiving Agent ---
def archive_doc(doc_name):
    with open("archived_docs.txt", "a") as f:
        f.write(f"{datetime.date.today()} - {doc_name}\n")


In [54]:
# --- Step 6: Audit Agent ---
audit_log = []
def log_action(doc_name, classification, explanation):
    audit_log.append({
        "timestamp": datetime.datetime.now().isoformat(),
        "document": doc_name,
        "classification": classification,
        "explanation": explanation
    })


In [55]:
# --- Step 7: Human-in-the-Loop Agent ---
def human_review(doc_name, doc_text):
    print(f"Manual review required for {doc_name}:")
    print(doc_text[:300], "...\n")
    decision = input("Enter classification (Cease/Irrelevant): ")
    return decision




In [56]:
# --- Step 8: Workflow ---
def workflow_stream(doc_path):
    doc_name = os.path.basename(doc_path)

    # Step 1: Load document
    yield f"📄 Loading document: {doc_name}"
    text, images = load_document(doc_path)

    # Step 2: Classification with confidence scores
    yield f"🔍 Running classification..."
    classification, confidences = classify_multimodal(text, images, return_confidence=True)

    # Stream confidence scores
    confidence_str = ", ".join([f"{lbl}: {sc:.2f}" for lbl, sc in confidences.items()])
    yield f"📊 Confidence scores → {confidence_str}"

    # Step 3: Recall prior context
    prior = db_tools.recall(doc_name)
    context_note = None
    if prior:
        context_note = f"Previously classified as {prior[4]} with human answer {prior[5]}"
        yield f"🗂️ Found prior context: {context_note}"

    # Step 4: Decision branches
    if classification == "Cease":
        db_tools.store(doc_name, text, "Cease", "Cease", conversation_context=context_note)
        yield f"✅ Stored {doc_name} as Cease"
    elif classification == "Irrelevant":
        archive_doc(doc_name)
        db_tools.store(doc_name, text, "Irrelevant", "Irrelevant", conversation_context=context_note)
        yield f"📦 Archived {doc_name} as Irrelevant"
    else:
        yield f"⚠️ Classification uncertain, escalating to human review..."
        human_answer = human_review(doc_name, text)
        db_tools.store(doc_name, text, "Uncertain", human_answer, conversation_context=context_note)
        yield f"🧑‍⚖️ Human review completed for {doc_name}: {human_answer}"


In [57]:
def generate_summary_report():
    headers, rows = db_tools.query_all()
    if not rows:
        print("No records found in the database.")
        return

    print("\n📊 Document Summary Report:")
    print(tabulate(rows, headers=headers, tablefmt="grid"))

    # Aggregate counts
    counts = {}
    for row in rows:
        doc_type = row[4]  # document_type column
        counts[doc_type] = counts.get(doc_type, 0) + 1

    print("\n📈 Classification Distribution:")
    for k, v in counts.items():
        print(f"{k}: {v}")

    # Show conversation context for each document
    print("\n🗂️ Conversation Context:")
    for row in rows:
        doc_name = row[2]
        context = row[7] if len(row) > 7 else None
        if context:
            print(f"{doc_name}: {context}")


In [58]:
# --- Step 10: Show Audit Trail ---
for entry in audit_log:
    print(entry)


In [59]:
# --- Robot Controller ---
db_tools = DBTools()
def robot_runner():
    db_tools.init_db()
    data_folder = "sample_data/docs"
    if not os.path.exists(data_folder):
        os.makedirs(data_folder)

    pdf_files = [f for f in os.listdir(data_folder) if f.endswith(".pdf")]
    if not pdf_files:
        print("No PDF files found in sample_data/docs.")
        return

    print(f"Found {len(pdf_files)} documents. Streaming results...\n")

    for file in pdf_files:
        print(f"\n--- Processing {file} ---")
        for update in workflow_stream(os.path.join(data_folder, file)):
            print(update)  # stream each step immediately

    print("\n📋 Audit Trail:")
    for entry in audit_log:
        print(entry)

    headers, rows = db_tools.query_all()
    print("\n📊 Database Records:")
    print(tabulate(rows, headers=headers, tablefmt="grid"))

    generate_summary_report()


# --- Run the robot ---
robot_runner()


Found 19 documents. Streaming results...


--- Processing 07_software_license_violation.pdf ---
📄 Loading document: 07_software_license_violation.pdf
🔍 Running classification...
📊 Confidence scores → Cease: 0.55, Uncertain: 0.37, Irrelevant: 0.29
✅ Stored 07_software_license_violation.pdf as Cease

--- Processing bw_doc_1.pdf ---
📄 Loading document: bw_doc_1.pdf
🔍 Running classification...
📊 Confidence scores → Uncertain: 0.72, Cease: 0.26, Irrelevant: 0.05
⚠️ Classification uncertain, escalating to human review...
Manual review required for bw_doc_1.pdf:

Abernathy & Rowe - Client Affairs

101 Meridian Avenue, Suite 210

Records Coordinator

Date: October 11, 2025

Notice Regarding Limited Power of Attorney Representation

We represent an individual referred to in this correspondence as ‘the Principal." We write

to advise that the Principal has exe ...

Enter classification (Cease/Irrelevant): cease
🧑‍⚖️ Human review completed for bw_doc_1.pdf: cease

--- Processing bw_doc_3.pdf ---
